# Tubes, vessels and fixed boundaries

This notebook shows the geometries and model factories for open tubes of
cells, and for flat tissues with fixed borders:

- `CylinderGeometry` with `model_factory_cylinder`: a tube around the z axis
  with surface elasticity and a lumen volume.
- `VesselGeometry` with `model_factory_vessel`: a tube held at a preferred
  radius, around any coordinate axis.
- `model_factory_bound`: a rectangular tissue whose border vertices do not move.

All three also use `ActiveMigration`.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

from tyssue import History, Sheet, SheetGeometry
from tyssue.draw import sheet_view
from tyssue.dynamics import effectors, model_factory_cylinder, model_factory_vessel
from tyssue.dynamics.factory import model_factory_bound
from tyssue.generation.hexagonal_grids import hexa_cylinder
from tyssue.generation.shapes import sheet_from_cell_centers
from tyssue.geometry.cylinder_geometry import CylinderGeometry
from tyssue.geometry.vessel_geometry import VesselGeometry
from tyssue.solvers.viscous import EulerSolver

## An open tube

`hexa_cylinder` spreads cell centers on a cylinder, `sheet_from_cell_centers`
builds the junction mesh, and we cut it to an open tube with
`extract_bounding_box`.

In [ ]:
def tube():
    centers = hexa_cylinder(12, 20, radius=2.0, capped=True)
    sheet = sheet_from_cell_centers(centers)
    sheet = sheet.extract_bounding_box(z_boundary=(-6.1, 6.1), coords=["x", "y", "z"])
    sheet.sanitize(trim_borders=False)
    return sheet


sheet = tube()
CylinderGeometry.update_all(sheet)
print(f"{sheet.Nf} cells, {sheet.vert_df['boundary'].sum()} border vertices")
fig, ax = sheet_view(sheet, coords=["z", "x"])

## `CylinderGeometry` and `model_factory_cylinder`

On top of `SheetGeometry`, `CylinderGeometry.update_all` computes for each
vertex the azimuthal tangent (`tx, ty, tz`), the distance to the z axis
(`distance_z_axis`) and the deviation from its neighbours (`dx, dy, dz` and
`dev_length`, used by `SurfaceElasticity`). It also stores the lumen volume in
`settings["lumen_vol"]`, closing the tube by a cone at each end.

`model_factory_cylinder` builds a model like `model_factory`, but holds the
tube in place: the bottom border vertices (z < 0) do not move, and the top
border vertices (z > 0) only move along z.

We make the cells in the upper part of the tube grow:

In [ ]:
model = model_factory_cylinder([
    effectors.LineTension,
    effectors.FaceAreaElasticity,
    effectors.PerimeterElasticity,
    effectors.SurfaceElasticity,
], effectors.FaceAreaElasticity)
sheet.update_specs(model.specs)

sheet.face_df["area_elasticity"] = 1.0
sheet.face_df["prefered_area"] = sheet.face_df["area"].mean()
sheet.face_df["perimeter_elasticity"] = 0.1
sheet.face_df["prefered_perimeter"] = sheet.face_df["perimeter"].mean()
sheet.edge_df["line_tension"] = 0.05
sheet.vert_df["viscosity"] = 1.0
sheet.vert_df["surface_elasticity"] = 1.0
sheet.vert_df["prefered_deviation"] = 0.0

sheet.face_df.loc[sheet.face_df["z"] > 2, "prefered_area"] *= 1.5
CylinderGeometry.update_all(sheet)
print("lumen volume before:", round(sheet.settings["lumen_vol"], 1))

solver = EulerSolver(sheet, CylinderGeometry, model)
solver.solve(tf=10, dt=0.05)
print("lumen volume after:", round(sheet.settings["lumen_vol"], 1))

top = sheet.vert_df["z"] > 2
bottom = sheet.vert_df["z"] < -2
print("mean radius, top / bottom:",
      round(sheet.vert_df.loc[top, "distance_z_axis"].mean(), 2),
      round(sheet.vert_df.loc[bottom, "distance_z_axis"].mean(), 2))
fig, ax = sheet_view(sheet, coords=["z", "x"], face={"visible": True, "color": sheet.face_df["area"]})

`CylinderGeometryInit`, the earlier name for this geometry, still works, with
a `FutureWarning`.

## `VesselGeometry` and a twisting vessel

`VesselGeometry.update_all` writes the distance of each vertex to the vessel
axis in `distance_origin`, and the unit radial vector in the two other
coordinates (`ox, oy` for the z axis). `VesselSurfaceElasticity` uses them to
hold each vertex at `prefered_radius`. `model_factory_vessel` keeps the bottom
border vertices from moving along z.

Here the cells in the upper half migrate around the tube, along the azimuthal
direction:

In [ ]:
vessel = tube()
VesselGeometry.update_all(vessel)

vmodel = model_factory_vessel([
    effectors.LineTension,
    effectors.FaceAreaElasticity,
    effectors.VesselSurfaceElasticity,
    effectors.ActiveMigration,
], effectors.FaceAreaElasticity)
vessel.update_specs(vmodel.specs)

vessel.face_df["area_elasticity"] = 1.0
vessel.face_df["prefered_area"] = vessel.face_df["area"].mean()
vessel.edge_df["line_tension"] = 0.05
vessel.vert_df["viscosity"] = 1.0
vessel.vert_df["vessel_elasticity"] = 5.0
vessel.vert_df["prefered_radius"] = vessel.vert_df["distance_origin"].mean()

# azimuthal migration direction, (x, y, 0) x z
fx, fy = vessel.face_df["x"].to_numpy(), vessel.face_df["y"].to_numpy()
radius = np.hypot(fx, fy)
vessel.face_df["mx"] = fy / radius
vessel.face_df["my"] = -fx / radius
vessel.face_df["mz"] = 0.0
vessel.face_df["migration_strength"] = np.where(vessel.face_df["z"] > 0, 0.3, 0.0)
VesselGeometry.update_all(vessel)

angle_before = np.arctan2(vessel.vert_df["y"], vessel.vert_df["x"])
EulerSolver(vessel, VesselGeometry, vmodel).solve(tf=10, dt=0.05)
angle_after = np.arctan2(vessel.vert_df["y"], vessel.vert_df["x"])
rotation = np.degrees(np.angle(np.exp(1j * (angle_after - angle_before))))

fig, ax = plt.subplots()
ax.scatter(vessel.vert_df["z"], rotation, s=8)
ax.set_xlabel("z")
ax.set_ylabel("rotation around the axis (degrees)")
print("radius: mean", round(vessel.vert_df["distance_origin"].mean(), 3),
      "std", round(vessel.vert_df["distance_origin"].std(), 3))

The vessel axis is set by `settings["axis"]` ("z" by default). The same tube
laid along x gives the same distances:

In [ ]:
along_x = tube()
along_x.vert_df[["x", "z"]] = along_x.vert_df[["z", "x"]].to_numpy()
along_x.settings["axis"] = "x"
VesselGeometry.update_all(along_x)

reference = tube()
VesselGeometry.update_all(reference)
np.allclose(along_x.vert_df["distance_origin"], reference.vert_df["distance_origin"])

## A tissue with fixed borders: `model_factory_bound`

`model_factory_bound` sets the gradient of every border vertex
(`vert_df["boundary"] == 1`) to zero, so the tissue border stays where it is.
A group of cells at the center migrates to the right:

In [ ]:
rect = Sheet.planar_sheet_3d("rect", 10, 10, 1, 1)
rect.sanitize()
SheetGeometry.update_all(rect)

bmodel = model_factory_bound([
    effectors.LineTension,
    effectors.FaceAreaElasticity,
    effectors.ActiveMigration,
], effectors.FaceAreaElasticity)
rect.update_specs(bmodel.specs, reset=True)

rect.face_df["prefered_area"] = rect.face_df["area"].mean()
rect.edge_df["line_tension"] = 0.05
rect.vert_df["viscosity"] = 1.0

center = rect.face_df[["x", "y"]].mean()
migrating = np.hypot(rect.face_df["x"] - center["x"], rect.face_df["y"] - center["y"]) < 2
rect.face_df["mx"], rect.face_df["my"], rect.face_df["mz"] = 1.0, 0.0, 0.0
rect.face_df["migration_strength"] = np.where(migrating, 0.5, 0.0)
SheetGeometry.update_all(rect)

border = rect.vert_df.index[rect.vert_df["boundary"] == 1]
border_before = rect.vert_df.loc[border, ["x", "y"]].to_numpy()

EulerSolver(rect, SheetGeometry, bmodel).solve(tf=10, dt=0.05)

print("border displacement:", np.abs(rect.vert_df.loc[border, ["x", "y"]].to_numpy() - border_before).max())
fig, ax = sheet_view(rect, face={"visible": True, "color": migrating.astype(float), "colormap": "coolwarm"})